# Test: L1-norm estimator variability

**Why `sample_sigma_as_pk` gives zero variability:**  
σ_ab = ∫(k/2π) P(k) W² dk is a smooth integral over many independent k-modes.
Per-mode cosmic variance is ~3–5%, but after summing ~100 modes the fractional
scatter in the integral is ~10⁻⁶ — essentially zero.

**The dominant source of L1 variability** is the **estimator variance** from
finite survey area.  For N_pix effectively independent pixels:

$$\operatorname{Var}(L_1) = \frac{\langle\kappa^2\rangle - \langle|\kappa|\rangle^2}{N_{\rm pix}}, \qquad N_{\rm pix} = \frac{A_{\rm survey}}{\pi\,\theta^2}$$

Both moments come directly from the LDT PDF — no P(k) sampling needed.
For a 10×10 deg² SLICS field at θ=15 arcmin this gives σ(L1)/L1 ≈ 3–5%,
matching the empirical scatter between SLICS realizations.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from wale.CosmologyModel import Cosmology_function
from wale.CovarianceMatrix import get_sigma_covariance, sample_sigma_as_pk
from wale.InitializeVariables import InitialiseVariables
from wale.VarianceCalculator import Variance
from wale.ComputePDF import computePDF
from wale.CommonUtils import (
    get_moments,
    compute_sigma_kappa_squared,
    get_l1_estimator_variance,
    sample_l1_from_pdf,
)

## 1. User parameters

In [ ]:
# --- Cosmology ---
h      = 0.681
Oc     = 0.2589
Ob     = 0.0486
w      = -1.0
wa     = 0.0
sigma8 = 0.817

# --- Angular scale and filter ---
theta1      = 15          # arcmin
filter_type = 'tophat'    # 'tophat' or 'starlet'

# --- n(z) file ---
tomo_bin = 5
nz_file  = f'../data/nz_arrays_bin{tomo_bin}.npy'

# --- k-grid ---
kmin = 1e-4
kmax = 1.0
dk   = 0.01

# --- Survey geometry for estimator variance ---
A_survey_deg2 = 100.0    # single SLICS tile = 10x10 deg2
# A_survey_deg2 = 15000.  # Euclid-like full survey

# --- SNR grid ---
snr = np.linspace(-6, 6, 200)

# --- Lambda range for saddle-point integration ---
lambdas = np.linspace(-450, 700, 20)

# --- Monte-Carlo mock maps ---
n_maps = 500   # number of mock survey realisations
seed   = 42

## 2. Initialise variables and run the fiducial LDT

In [ ]:
variables = InitialiseVariables(
    h=h, Oc=Oc, Ob=Ob, w=w, wa=wa, sigma8=sigma8,
    dk=dk, kmin=kmin, kmax=kmax,
    nz_file=nz_file,
    variability=False,
    theta1=theta1,
    nplanes=8,
    numberofrealisations=1,
)
k = variables.cosmo.k
print(f'nz planes : {len(variables.redshifts)}')
print(f'theta1    : {theta1} arcmin  ({variables.theta1_radian:.5f} rad)')

In [ ]:
variance = Variance(variables.cosmo, filter_type=filter_type, pk=variables.cosmo.pnl)

sigmasq_ldt = np.sum(
    variables.dchi * (variables.lensingweights ** 2) * np.array([
        variance.get_sig_slice(z_i, chi * variables.theta1_radian, chi * variables.theta2_radian)
        for z_i, chi in zip(variables.redshifts, variables.chis)
    ])
)

sigmasq_pt = compute_sigma_kappa_squared(
    theta1, variables.chis, variables.lensingweights,
    variables.redshifts, k, variables.cosmo.pnl,
    filter_type=filter_type, h=h,
)

variables.recal_value = sigmasq_ldt / sigmasq_pt
variables.sigmasq_map = sigmasq_ldt
variables.lambdas     = lambdas

print(f'LDT sigma2_kappa : {sigmasq_ldt:.6e}')
print(f'PT  sigma2_kappa : {sigmasq_pt:.6e}')
print(f'Recal factor     : {variables.recal_value:.6f}')

computed_PDF = computePDF(variables, variance, plot_scgf=False)
pdf_vals     = computed_PDF.pdf_values
kappa_vals   = computed_PDF.kappa_values
norm_kappa   = kappa_vals / np.sqrt(sigmasq_pt)

l1_fid  = CubicSpline(norm_kappa, np.abs(kappa_vals) * pdf_vals)(snr)
pdf_fid = CubicSpline(norm_kappa, pdf_vals)(snr)
print(f'L1 peak value    : {l1_fid.max():.4e}')

## 3. Analytical L1 estimator variance

No sampling — computed directly from the LDT PDF.

In [ ]:
l1_mean_anal, l1_std_anal, n_pix_eff = get_l1_estimator_variance(
    pdf_vals, kappa_vals, A_survey_deg2, theta1
)

print(f'Survey area   : {A_survey_deg2:.0f} deg2')
print(f'N_pix_eff     : {n_pix_eff:.0f}')
print(f'<|kappa|>     : {l1_mean_anal:.4e}')
print(f'sigma(L1)     : {l1_std_anal:.4e}  ({100*l1_std_anal/l1_mean_anal:.2f}%)')

In [ ]:
# How sigma(L1) scales with survey area
areas = np.array([10., 25., 100., 400., 1000., 5000., 15000.])
stds  = [get_l1_estimator_variance(pdf_vals, kappa_vals, A, theta1)[1] for A in areas]

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(areas, stds, 'o-', color='steelblue')
ax.loglog(areas, stds[0] * np.sqrt(areas[0] / areas), 'k--', alpha=0.5, label='prop 1/sqrt(A)')
ax.axvline(A_survey_deg2, color='tomato', ls=':', label=f'A={A_survey_deg2} deg2')
ax.set_xlabel('Survey area [deg2]')
ax.set_ylabel('sigma(L1)')
ax.set_title(f'L1 estimator std vs survey area  (theta={theta1} arcmin)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Monte-Carlo sampling distribution of L1

Draw `n_maps` mock surveys — each draws `n_pix_eff` independent kappa values from the LDT PDF.

In [ ]:
print(f'Drawing {n_maps} mock surveys with N_pix={int(n_pix_eff)} pixels each...')
l1_mc, n_pix_used = sample_l1_from_pdf(
    pdf_vals, kappa_vals,
    A_survey_deg2=A_survey_deg2,
    theta_arcmin=theta1,
    n_maps=n_maps,
    seed=seed,
)
print('Done.')
print(f'MC  <L1>={l1_mc.mean():.4e}  sigma(L1)={l1_mc.std():.4e}  ({100*l1_mc.std()/l1_mc.mean():.2f}%)')
print(f'Analytical  <L1>={l1_mean_anal:.4e}  sigma(L1)={l1_std_anal:.4e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(l1_mc, bins=40, color='steelblue', alpha=0.7, density=True)
ax.axvline(l1_mc.mean(),  color='steelblue', lw=2, label=f'MC mean={l1_mc.mean():.3e}')
ax.axvline(l1_mean_anal, color='k', lw=2, ls='--', label=f'LDT <|k|>={l1_mean_anal:.3e}')
ax.axvspan(l1_mean_anal - l1_std_anal, l1_mean_anal + l1_std_anal,
           color='tomato', alpha=0.2, label='Analytical +/-1 sigma')
ax.set_xlabel('L1 norm  <|kappa|>')
ax.set_ylabel('Probability density')
ax.set_title(f'L1 sampling distribution  ({n_maps} mocks, A={A_survey_deg2} deg2)')
ax.legend(fontsize=8)

ax = axes[1]
# Schematic band scaled by local |kappa|P(kappa) weight
weight = np.abs(snr) * np.maximum(pdf_fid, 0)
w_max  = weight.max()
band   = l1_std_anal * (weight / w_max if w_max > 0 else np.ones_like(weight)) * l1_fid.max()
ax.plot(snr, l1_fid, 'k-', lw=2, label='Fiducial LDT')
ax.fill_between(snr, l1_fid - band, l1_fid + band,
                color='steelblue', alpha=0.4, label=f'+/-1sigma  (A={A_survey_deg2:.0f} deg2)')
ax.set_xlabel('kappa / sigma_kappa')
ax.set_ylabel('|kappa| P(kappa)')
ax.set_title('Fiducial L1 with estimator variance band')
ax.legend()

plt.suptitle(
    f'h={h}, sigma8={sigma8}, Om={Oc+Ob:.4f}  |  theta={theta1} arcmin, {filter_type}, bin {tomo_bin}',
    fontsize=10
)
plt.tight_layout()
plt.show()

## 5. Why P(k) cosmic variance is negligible

The sigma-space covariance captures the scatter in sigma_ab between box realisations.
Because sigma_ab integrates over ~100 independent k-modes, per-mode 3-5% scatter
averages down to ~1e-6 fractional variance — compare to ~3-5% from the estimator variance.

In [ ]:
s_mean, C_sigma = get_sigma_covariance(
    variables.cosmo, variables.redshifts,
    variables.theta1_radian, variables.theta2_radian,
    filter_type=filter_type,
    f_sky=A_survey_deg2 / 41253.0,
)

sig11_fid = s_mean[0::3]
sig11_std = np.sqrt(np.diag(C_sigma)[0::3])

print('P(k) cosmic variance contribution to sigma11 per z-slice:')
for z_i, m, s in zip(variables.redshifts, sig11_fid, sig11_std):
    print(f'  z={z_i:.3f}  frac_std(sigma11) = {s/m:.2e}')

print(f'\nEstimator frac_std(L1) = {l1_std_anal/l1_mean_anal:.2e}')
print(f'Ratio (estimator / P(k) CV) = {(l1_std_anal/l1_mean_anal) / (sig11_std/sig11_fid).mean():.0f}x larger')

In [ ]:
# Run LDT for a handful of P(k) samples — should be invisible against the estimator band
n_pk = 5
pk_samples, pk_fid_dict = sample_sigma_as_pk(
    variables.cosmo, variables.redshifts, s_mean, C_sigma, n_samples=n_pk, seed=seed
)

l1_pk_runs = []
for powspec in [pk_fid_dict] + pk_samples:
    var_n  = Variance(variables.cosmo, filter_type=filter_type, pk=powspec)
    sig_n  = np.sum(
        variables.dchi * (variables.lensingweights ** 2) * np.array([
            var_n.get_sig_slice(z_i, chi * variables.theta1_radian, chi * variables.theta2_radian)
            for z_i, chi in zip(variables.redshifts, variables.chis)
        ])
    )
    spt_n  = compute_sigma_kappa_squared(
        theta1, variables.chis, variables.lensingweights,
        variables.redshifts, k, powspec, filter_type=filter_type, h=h,
    )
    variables.recal_value = sig_n / spt_n
    variables.sigmasq_map = sig_n
    pdfn  = computePDF(variables, var_n, plot_scgf=False)
    nk_n  = pdfn.kappa_values / np.sqrt(spt_n)
    l1_pk_runs.append(CubicSpline(nk_n, np.abs(pdfn.kappa_values) * pdfn.pdf_values)(snr))

l1_pk_runs = np.array(l1_pk_runs)

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(snr, l1_fid - band, l1_fid + band,
                color='steelblue', alpha=0.35, label=f'Estimator +/-1sigma  (A={A_survey_deg2:.0f} deg2)')
for i in range(1, len(l1_pk_runs)):
    ax.plot(snr, l1_pk_runs[i], color='tomato', alpha=0.8, lw=1,
            label='P(k) sample' if i == 1 else '')
ax.plot(snr, l1_fid, 'k-', lw=2, label='Fiducial')
ax.set_xlabel('kappa / sigma_kappa')
ax.set_ylabel('|kappa| P(kappa)')
ax.set_title('Estimator band (blue) vs P(k) CV samples (red)\nP(k) samples are invisible against the estimator band')
ax.legend()
plt.tight_layout()
plt.show()

pk_spread = np.std(l1_pk_runs[1:], axis=0).max()
print(f'Max L1 spread from P(k) CV : {pk_spread:.2e}')
print(f'Estimator sigma(L1)        : {l1_std_anal:.2e}')
print(f'Ratio                      : {l1_std_anal/pk_spread:.0f}x larger')